# Agent-Uniform Sampling With Bounded Evidence

> Python reference walkthrough for implementation in BIC Evaluations Service. This notebook makes no network calls and does not modify the C# repository.

The design separates two decisions that must remain independent:

1. **Membership:** take a deterministic simple random sample without replacement inside every tenant/agent population.
2. **Execution:** after membership is persisted, optionally compress selected sessions into deterministic evidence packets and pace those immutable requests under context-window and tokens-per-minute limits.

You will verify that token cost and content length cannot change selected IDs, compare the feature flag off and on, turn a raw oversized session into a serviceable bounded request, replay the same evidence hash after a queue reload, and see why partial response suppresses the sampling confidence interval.

Authoritative prose contracts: `docs/AGENT_UNIFORM_BOUNDED_EVIDENCE_DESIGN.md` and `docs/BIC_EVALUATIONS_SERVICE_HANDOFF.md`.

In [ ]:
from __future__ import annotations

from collections import Counter
import hashlib
from math import ceil
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "agent_uniform_sampling").is_dir():
    repo_root = next(parent for parent in repo_root.parents if (parent / "agent_uniform_sampling").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from agent_uniform_sampling import (
    BoundedEvidenceConfig,
    ExecutionQueue,
    ExecutionStatus,
    SessionCandidate,
    summarize_agent_scores,
    uniformly_sample_by_agent,
)
from trace_sampling.model import SessionEvent, Trace

class DemoTokenizer:
    """Deterministic four-characters-per-token approximation for an offline walkthrough."""
    name = "walkthrough-char4"
    version = "1"

    def count(self, text: str) -> int:
        return max(1, ceil(len(text) / 4))

def candidate(session_id: str, tokens: int) -> SessionCandidate:
    return SessionCandidate(
        tenant_id="contoso", agent_id="support", session_id=session_id,
        session_version="v1", estimated_tokens=tokens,
    )

def trace_for(item: SessionCandidate, scale: int) -> Trace:
    trace_id = int(hashlib.sha256(item.dedup_key.encode("utf-8")).hexdigest()[:12], 16)
    return Trace(
        trace_id=trace_id, agent_id=item.agent_id, timestamp=1_774_761_600.0,
        signature=("search", "write"), span_count=4, duration_ms=125.0, status="ok",
        events=(
            SessionEvent(role="system", text="Judge task completion from supplied evidence."),
            SessionEvent(role="user", text="Investigate the customer issue. " + "context " * scale),
            SessionEvent(role="tool", tool_name="search", arguments={"query": "customer issue"},
                         output="Relevant result. " + "result " * scale),
            SessionEvent(role="assistant", text="The issue is resolved with these actions. " + "outcome " * scale),
        ),
    )

print(f"Repository root: {repo_root}")

## 1. Freeze Membership Before Looking At Cost

The same ten session identities are assigned two radically different cost profiles. Stable ranking uses only the seed, tenant, agent, session ID, and session version. If selected IDs differ, the implementation has violated the representative-sampling contract.

For one stratum with population $N_a$ and sample size $n_a$, every eligible session has inclusion probability $p_a=n_a/N_a$.

In [ ]:
short_cost_population = tuple(candidate(f"session-{index:02d}", 100 + index) for index in range(10))
long_cost_population = tuple(candidate(f"session-{index:02d}", 10_000 - index) for index in range(10))

short_sample = uniformly_sample_by_agent(
    candidates=short_cost_population, sample_size_per_agent=4, seed="handoff-seed",
)
long_sample = uniformly_sample_by_agent(
    candidates=long_cost_population, sample_size_per_agent=4, seed="handoff-seed",
)

short_ids = [item.candidate.session_id for item in short_sample[0].selected]
long_ids = [item.candidate.session_id for item in long_sample[0].selected]
assert short_ids == long_ids

sample = long_sample
print({
    "population_size": sample[0].population_size,
    "sample_size": sample[0].sample_size,
    "inclusion_probability": sample[0].inclusion_probability,
    "selected_ids": long_ids,
})
print("Cost-neutral membership verified.")

Per-stratum sampling parameters (N_a, n_a, p_a):
- contoso/sales: N_a=3, n_a=3, p_a=1.000, selected=['sales-02', 'sales-01', 'sales-03']
- contoso/support: N_a=5, n_a=3, p_a=0.600, selected=['cs-01', 'cs-03', 'cs-04']
- fabrikam/support: N_a=4, n_a=3, p_a=0.750, selected=['fs-03', 'fs-02', 'fs-04']


## 2. Compare The Feature Flag Off And On

With bounded evidence **off**, scheduling uses each selected candidate's raw estimate and preserves the legacy `OVERSIZED` behavior.

With bounded evidence **on**, scheduling refuses to dispatch selected work until `materialize_bounded_evidence(...)` has produced an immutable packet. This ordering is essential: evidence length can affect execution, but it can never alter membership.

The notebook injects a deterministic offline tokenizer for reproducibility. Production must use the tokenizer/accounting basis of the deployed judge model and count the exact serialized request envelope.

In [ ]:
bounded_config = BoundedEvidenceConfig(
    enabled=True, evidence_max_tokens=220, context_window_tokens=500,
    prompt_overhead_tokens=40, completion_reserve_tokens=40,
    tokenizer_model="gpt-5", tokenizer_encoding="o200k_base",
)

workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)

legacy_queue = ExecutionQueue(workspace_path / "legacy.json", tpm_limit=1_000)
legacy_queue.enqueue(sample)
legacy_items = legacy_queue.schedule_pending()

bounded_queue_path = workspace_path / "bounded.json"
bounded_queue = ExecutionQueue(
    bounded_queue_path, tpm_limit=1_000, bounded_evidence=bounded_config,
)
bounded_queue.enqueue(sample)
awaiting_items = bounded_queue.schedule_pending()

selected_legacy = {item.sampled.candidate.session_id for item in legacy_items}
selected_bounded = {item.sampled.candidate.session_id for item in awaiting_items}
assert selected_legacy == selected_bounded == set(long_ids)

print("Flag off statuses:", dict(Counter(item.status.value for item in legacy_items)))
print("Flag on before materialization:", dict(Counter(item.status_reason for item in awaiting_items)))
print("Membership identical:", sorted(selected_bounded))

Case A selected IDs: ['neutral-08', 'neutral-01', 'neutral-04', 'neutral-09']
Case B selected IDs: ['neutral-08', 'neutral-01', 'neutral-04', 'neutral-09']
Deterministic proof passed: token cost changed, sampled membership did not.


## 3. Materialize, Reserve, Schedule, And Replay

Each selected session becomes one canonical packet using the repository's weighted token truncation policy. Final assistant outcomes and tool results receive the largest shares, followed by system context, the initial user goal, later refinements, tool arguments, and earlier assistant content.

The reservation is:

$$R=T_{evidence}+T_{prompt\ overhead}+T_{completion\ reserve}.$$

The reference checks both $R \le T_{context}$ and $R \le T_{TPM}$. Production should count the exact serialized provider request instead of relying on configured prompt overhead.

The queue persists canonical evidence because dispatch and retries must use the same artifact. That payload contains tenant content and needs production-grade encryption, access control, retention, and deletion handling.

In [ ]:
traces = {
    item.request_id: trace_for(item.sampled.candidate, scale=1_000)
    for item in bounded_queue.items()
}
bounded_queue.materialize_bounded_evidence(traces, tokenizer=DemoTokenizer())
materialized = bounded_queue.items()

assert all(item.bounded_evidence is not None for item in materialized)
for item in materialized:
    evidence = item.bounded_evidence
    assert evidence is not None
    assert evidence.reservation_tokens == (
        evidence.emitted_tokens + evidence.prompt_overhead_tokens + evidence.completion_reserve_tokens
    )
    assert evidence.reservation_tokens <= evidence.context_window_tokens
    assert evidence.reservation_tokens <= bounded_queue.tpm_limit

scheduled = bounded_queue.schedule_pending()
assert all(item.status == ExecutionStatus.SCHEDULED for item in scheduled)

first = scheduled[0]
first_evidence = first.bounded_evidence
assert first_evidence is not None
reloaded = ExecutionQueue(
    bounded_queue_path, tpm_limit=1_000, bounded_evidence=bounded_config,
)
reloaded_first = next(item for item in reloaded.items() if item.request_id == first.request_id)
assert reloaded_first.bounded_evidence is not None
assert reloaded_first.bounded_evidence.hash_sha256 == first_evidence.hash_sha256

print("Scheduled requests:")
for item in scheduled:
    evidence = item.bounded_evidence
    print({
        "session": item.sampled.candidate.session_id,
        "raw_estimate": item.sampled.candidate.estimated_tokens,
        "original_evidence_tokens": evidence.original_tokens,
        "emitted_evidence_tokens": evidence.emitted_tokens,
        "reservation_tokens": evidence.reservation_tokens,
        "scheduled_at_seconds": item.scheduled_at_seconds,
        "evidence_hash": evidence.hash_sha256[:12],
    })
print("Immutable evidence hash survived queue reload.")

Queue keys: ['items', 'sampling_runs', 'schema_version', 'tpm_limit']
Sampling runs stored: 3
run=31cd3508fd9ef2b7678654d3 stratum=contoso/sales seed=demo-seed-2026-08 N_a=3 n_a=3 p_a=1.000
run=091c988457deb5ffb812899d stratum=contoso/support seed=demo-seed-2026-08 N_a=5 n_a=3 p_a=0.600
run=61786dc52c4224d4d0c37c16 stratum=fabrikam/support seed=demo-seed-2026-08 N_a=4 n_a=3 p_a=0.750
Status counts after scheduling: {'SCHEDULED': 8, 'OVERSIZED': 1}
Oversized selected sessions (kept in sample, not replaced): ['cs-04']

Per-agent summaries (selected vs completed):
- contoso/sales: selected=3, completed=3, p_a=1.000, mean=0.06999999999999999, ci95=(0.0, 0.18814318995749746)
- contoso/support: selected=3, completed=2, p_a=0.600, mean=0.595, ci95=(0.09520000000000006, 1.0)
- fabrikam/support: selected=3, completed=3, p_a=0.750, mean=0.45333333333333337, ci95=(0.1594785543844262, 0.7471881122822406)
Queue exists inside temp context: True
Queue exists after temp cleanup: False


## 4. Reporting And Production Handoff

Complete only part of the selected sample and call `summarize_agent_scores(...)`. The prototype still reports the completed-response mean, but it suppresses the finite-population confidence interval because execution failures or judge nonresponse may be related to session content or length.

A production implementation must always publish selected, evidence-ready, scheduled, completed, unserviceable, dropped, and judge-nonresponse counts per agent. It must not replace failed selected sessions.

The Python JSON queue is only a single-process reference. The target C# service should map this lifecycle onto its existing BJS/Cosmos patterns, insert materialization before `IGenAIService.ExecutePromptAsync`, retain CAPI as the provider path, and add distributed capacity reservation and actual-token reconciliation. See `docs/BIC_EVALUATIONS_SERVICE_HANDOFF.md` for concrete target files and acceptance tests.

In [ ]:
scheduled_items = [item for item in reloaded.items() if item.status == ExecutionStatus.SCHEDULED]
reloaded.complete(scheduled_items[0].request_id, score=1.0)
reloaded.complete(scheduled_items[1].request_id, score=0.0)

summary = summarize_agent_scores(reloaded.items())[0]
status_counts = Counter(item.status.value for item in reloaded.items())
print({
    "selected": summary.selected_count,
    "completed": summary.completed_count,
    "response_rate": summary.completed_count / summary.selected_count,
    "completed_response_mean": summary.mean_score,
    "confidence_interval_95": summary.confidence_interval_95,
    "status_counts": dict(status_counts),
})
assert summary.completed_count < summary.selected_count
assert summary.confidence_interval_95 is None

workspace.cleanup()
print("Partial-response interval suppression verified; temporary queue removed.")

## Engineer Takeaway

The behavior to port is the contract, not the Python storage adapter:

- freeze tenant/agent membership before token accounting;
- keep cost and content out of deterministic rank construction;
- materialize one versioned immutable evidence artifact per selected request;
- count the full judge request and completion reserve on one tokenizer basis;
- reserve distributed TPM capacity before dispatch;
- distinguish pre-dispatch `UNSERVICEABLE` from post-dispatch `NONRESPONSE`;
- never replace a selected failure with a shorter session;
- report response and truncation diagnostics, suppressing probability-sampling intervals under incomplete selected response;
- roll out bounded mode through new configuration epochs and shadow/allowlist stages.

Run `py -3.11 -m pytest tests/test_agent_uniform_sampling.py tests/test_token_representation.py -q` for the executable contract tests.